## Example 1: botocore functionality test 1

In [ ]:
import botocore
from botocore.session import Session
from botocore.exceptions import NoCredentialsError

# Example 1: Basic botocore session and client creation
print("=== Botocore Basic Session and Client Creation ===")

try:
    # Show botocore version
    print(f"Botocore version: {botocore.__version__}")
    
    # Create a botocore session (lower-level than boto3)
    session = Session()
    print(f"Botocore session created: {type(session).__name__}")
    
    # Show session capabilities
    available_services = session.get_available_services()
    print(f"Available AWS services: {len(available_services)}")
    print(f"Sample services: {available_services[:8]}")
    
    # Create a low-level client
    try:
        client = session.create_client('s3', region_name='us-east-1')
        print(f"S3 client created: {type(client).__name__}")
        print(f"Client service model: {client._service_model.service_name}")
        print(f"API version: {client._service_model.api_version}")
    except Exception as client_error:
        print(f"Client creation note: {type(client_error).__name__}")
        raise
    
    # Show available regions
    available_regions = session.get_available_regions('ec2')
    print(f"EC2 available in {len(available_regions)} regions")
    print(f"Sample regions: {available_regions[:6]}")
    
    # Show profile information
    available_profiles = session.available_profiles
    print(f"Available credential profiles: {available_profiles}")
    
    print("Basic botocore session creation completed successfully!")
    
except Exception as e:
    print(f"Error in basic botocore functionality: {e}")
    raise AssertionError(
        f"botocore example (cell 1) failed: {type(e).__name__}: {e}"
    ) from e

## Example 2: botocore functionality test 2

In [ ]:
import botocore
from botocore.session import Session
from botocore.loaders import Loader
import json

# Example 2: Service models and API introspection with botocore
print("=== Botocore Service Models and API Introspection ===")

try:
    # Create session and client
    session = Session()
    client = session.create_client('s3', region_name='us-east-1')
    
    # Access service model directly
    service_model = client._service_model
    print(f"Service model: {service_model.service_name}")
    print(f"Protocol: {service_model.protocol}")
    print(f"API version: {service_model.api_version}")
    print(f"Endpoint prefix: {service_model.endpoint_prefix}")
    
    # Explore available operations
    operations = service_model.operation_names
    print(f"\nTotal S3 operations: {len(operations)}")
    print(f"Sample operations: {operations[:8]}")
    
    # Examine specific operation model
    list_buckets_op = service_model.operation_model('ListBuckets')
    print(f"\nListBuckets operation:")
    print(f"  HTTP method: {list_buckets_op.http.get('method', 'GET')}")
    print(f"  HTTP URI: {list_buckets_op.http.get('requestUri', '/')}")
    print(f"  Documentation: {list_buckets_op.documentation[:100] if list_buckets_op.documentation else 'None'}...")
    
    # Examine operation with parameters
    get_object_op = service_model.operation_model('GetObject')
    print(f"\nGetObject operation:")
    print(f"  HTTP method: {get_object_op.http.get('method', 'GET')}")
    
    if get_object_op.input_shape:
        required_params = list(get_object_op.input_shape.required_members)
        print(f"  Required parameters: {required_params}")
        
        # Show parameter details
        input_members = get_object_op.input_shape.members
        print(f"  Total parameters: {len(input_members)}")
        print(f"  Sample parameters: {list(input_members.keys())[:5]}")
    
    # Use botocore loader to get raw service definitions
    print(f"\n=== Service Definition Loading ===")
    loader = Loader()
    api_model = loader.load_service_model('s3', 'service-2', '2006-03-01')
    
    print(f"Raw API model loaded")
    print(f"Operations in raw model: {len(api_model.get('operations', {}))}")
    print(f"Service metadata: {list(api_model.get('metadata', {}).keys())}")
    
    print("Service models and API introspection completed successfully!")
    
except Exception as e:
    print(f"Error in service model introspection: {e}")
    raise AssertionError(
        f"botocore example (cell 3) failed: {type(e).__name__}: {e}"
    ) from e

## Example 3: botocore functionality test 3

In [ ]:
import botocore
from botocore.session import Session
from botocore.exceptions import ClientError
from botocore.stub import Stubber
from botocore.config import Config
import time

# Example 3: Configuration, retry logic, and error handling
print("=== Botocore Configuration and Error Handling ===")

try:
    # Custom configuration with advanced options
    config = Config(
        region_name='us-west-2',
        retries={
            'max_attempts': 5,
            'mode': 'adaptive'
        },
        connect_timeout=15,
        read_timeout=30,
        max_pool_connections=25,
        parameter_validation=True,
        signature_version='v4'
    )
    
    print(f"Custom config created:")
    print(f"  Region: {config.region_name}")
    print(f"  Max retry attempts: {config.retries['max_attempts']}")
    print(f"  Retry mode: {config.retries['mode']}")
    print(f"  Connect timeout: {config.connect_timeout}s")
    print(f"  Read timeout: {config.read_timeout}s")
    
    # Create session with custom config
    session = Session()
    client = session.create_client('s3', config=config)
    
    print(f"\nClient created with custom configuration")
    print(f"Client region: {client.meta.region_name}")
    
    # Demonstrate error handling patterns
    print(f"\n=== Error Handling Patterns ===")
    
    # botocore's Stubber answers the call locally with a canned AccessDenied error, so it never
    # reaches AWS and behaves the same with or without credentials or network access.
    stubber = Stubber(client)
    stubber.add_client_error(
        'list_buckets',
        service_error_code='AccessDenied',
        service_message='Access Denied',
        http_status_code=403,
    )
    with stubber:
        try:
            client.list_buckets()
        except ClientError as e:
            error_code = e.response['Error']['Code']
            http_status = e.response['ResponseMetadata']['HTTPStatusCode']
            print(f"✓ ClientError caught: {error_code}")
            print(f"  HTTP status: {http_status}")
        else:
            raise AssertionError("the stubbed list_buckets call did not raise ClientError")
    stubber.assert_no_pending_responses()
    assert (error_code, http_status) == ('AccessDenied', 403), (error_code, http_status)
    
    # Show built-in retry configuration
    print(f"\n=== Retry Configuration Details ===")
    retry_config = client._client_config.retries
    print(f"Retry configuration: {retry_config}")
    
    # Demonstrate endpoint resolver
    print(f"\n=== Endpoint Resolution ===")
    endpoint_resolver = session.get_component('endpoint_resolver')
    
    # construct_endpoint() returns the endpoint's metadata; the host is under 'hostname' (there is no
    # 'endpoint_url' key).
    s3_endpoint = endpoint_resolver.construct_endpoint('s3', 'us-east-1')
    assert s3_endpoint['partition'] == 'aws' and s3_endpoint['hostname'].startswith('s3.'), s3_endpoint
    print(f"S3 us-east-1 endpoint: https://{s3_endpoint['hostname']}")
    
    ec2_endpoint = endpoint_resolver.construct_endpoint('ec2', 'eu-west-1')
    assert ec2_endpoint['hostname'] == 'ec2.eu-west-1.amazonaws.com', ec2_endpoint
    print(f"EC2 eu-west-1 endpoint: https://{ec2_endpoint['hostname']}")
    
    # Show credential resolution
    print(f"\n=== Credential Resolution ===")
    # The session registers the resolver under 'credential_provider' ('credential_resolver' is not a
    # component name).
    credential_resolver = session.get_component('credential_provider')
    print(f"Credential resolver: {type(credential_resolver).__name__}")
    print("Credential provider chain order:")
    
    providers = credential_resolver.providers
    provider_names = [getattr(provider, 'METHOD', type(provider).__name__) for provider in providers]
    for i, provider_name in enumerate(provider_names, 1):
        print(f"  {i}. {provider_name}")
    assert provider_names[0] == 'env' and 'shared-credentials-file' in provider_names, provider_names
    
    # Show session configuration
    print(f"\n=== Session Configuration ===")
    session_config = session.get_scoped_config()
    print(f"Session config keys: {list(session_config.keys()) if session_config else 'None'}")
    
    if session_config:
        # Show non-sensitive config items
        safe_keys = ['region', 'output', 'cli_pager']
        for key in safe_keys:
            if key in session_config:
                print(f"  {key}: {session_config[key]}")
    
    print("Configuration and error handling demonstration completed!")
    
except Exception as e:
    print(f"Error in configuration demonstration: {e}")
    raise AssertionError(
        f"botocore example (cell 5) failed: {type(e).__name__}: {e}"
    ) from e

## Example 4: botocore functionality test 4

In [ ]:
import botocore
from botocore.session import Session
from botocore.paginate import Paginator
from botocore.waiter import WaiterModel
from botocore import xform_name
import json

# Example 4: Advanced features - Paginators, Waiters, and Event System
print("=== Botocore Advanced Features: Paginators and Waiters ===")

try:
    # Create session and client
    session = Session()
    s3_client = session.create_client('s3', region_name='us-east-1')
    
    # Demonstrate paginator functionality
    print("=== Paginator Functionality ===")
    
    # Get paginator for list_objects_v2
    paginator = s3_client.get_paginator('list_objects_v2')
    print(f"List objects paginator created: {type(paginator).__name__}")
    
    # Show paginator configuration: a Paginator has no `paginate_config`; the operation's pagination
    # model comes from the session, and the paginator exposes its result keys.
    paginate_config = session.get_paginator_model('s3').get_paginator('ListObjectsV2')
    print(f"Pagination config keys: {list(paginate_config.keys())}")
    assert paginate_config['input_token'] == 'ContinuationToken', paginate_config
    assert [key.expression for key in paginator.result_keys] == paginate_config['result_key'], paginator.result_keys
    
    # Show available paginators for S3 (a bound method has no `_service_model`; ask the client)
    paginator_names = [xform_name(op) for op in s3_client.meta.service_model.operation_names
                       if s3_client.can_paginate(xform_name(op))]
    assert 'list_objects_v2' in paginator_names, paginator_names
    print(f"Available S3 paginators: {len(paginator_names)}")
    print(f"Sample paginators: {paginator_names[:6]}")
    
    # Demonstrate waiter functionality
    print(f"\n=== Waiter Functionality ===")
    
    # Get waiter for bucket_exists
    waiter = s3_client.get_waiter('bucket_exists')
    print(f"Bucket exists waiter created: {type(waiter).__name__}")
    
    # Show waiter configuration: a Waiter has no `_waiter_model`; the service's model comes from the session
    waiter_model = session.get_waiter_model('s3')
    print(f"Waiter model: {type(waiter_model).__name__}")
    
    # List available waiters
    available_waiters = s3_client.waiter_names
    print(f"Available S3 waiters: {available_waiters}")
    assert 'bucket_exists' in available_waiters, available_waiters
    
    # Show waiter details (the model is keyed by the CamelCase name; the waiter carries the same config)
    waiter_config = waiter_model.get_waiter('BucketExists')
    assert (waiter_config.delay, waiter_config.max_attempts) == (waiter.config.delay, waiter.config.max_attempts)
    print(f"Bucket exists waiter delay: {waiter_config.delay}s")
    print(f"Bucket exists waiter max attempts: {waiter_config.max_attempts}")
    
    # Demonstrate event system
    print(f"\n=== Event System ===")
    
    # Access the event emitter
    event_emitter = s3_client.meta.events
    print(f"Event emitter: {type(event_emitter).__name__}")
    
    # Example event handler
    def log_api_call(event_name, **kwargs):
        print(f"Event fired: {event_name}")
    
    # Register event handler (demonstration only)
    event_name = 'before-call.s3.ListBuckets'
    event_emitter.register(event_name, log_api_call)
    print(f"Registered event handler for: {event_name}")
    
    # Show some common event names
    common_events = [
        'before-call',
        'after-call', 
        'before-parameter-build',
        'after-parameter-build',
        'before-sign',
        'before-send'
    ]
    print(f"Common event patterns: {common_events}")
    
    # EC2 waiter example
    print(f"\n=== EC2 Waiter Example ===")
    ec2_client = session.create_client('ec2', region_name='us-east-1')
    ec2_waiters = ec2_client.waiter_names
    print(f"EC2 available waiters: {len(ec2_waiters)}")
    print(f"Sample EC2 waiters: {ec2_waiters[:6]}")
    
    print("Advanced features demonstration completed successfully!")
    
except Exception as e:
    print(f"Error in advanced features demonstration: {e}")
    raise AssertionError(
        f"botocore example (cell 7) failed: {type(e).__name__}: {e}"
    ) from e

## Example 5: botocore functionality test 5

In [ ]:
import botocore
from botocore.session import Session
from botocore.awsrequest import AWSRequest
from botocore.auth import SigV4Auth
import json
import sys

# Example 5: Comprehensive botocore architecture and low-level capabilities
print("=== Botocore Comprehensive Architecture Overview ===")

try:
    print(f"Botocore version: {botocore.__version__}")
    print(f"Python version: {sys.version_info.major}.{sys.version_info.minor}")
    
    # Show botocore's role as foundation
    print(f"\n=== Botocore as AWS SDK Foundation ===")
    print("• Low-level AWS API client library")
    print("• Foundation for boto3 high-level interface") 
    print("• Direct protocol and service model handling")
    print("• Comprehensive AWS service coverage")
    print("• Credential and configuration management")
    
    # Session architecture
    print(f"\n=== Session Architecture Components ===")
    session = Session()
    
    # Core session components, by the names the session registers them under ('credential_resolver' and
    # 'client_creator' are not component names: get_component raises ValueError for both).
    components = [
        'credential_provider',
        'data_loader', 
        'endpoint_resolver',
        'event_emitter',
        'response_parser_factory'
    ]
    
    for component_name in components:
        component = session.get_component(component_name)
        print(f"• {component_name}: {type(component).__name__}")
    
    # Service coverage
    print(f"\n=== AWS Service Coverage ===")
    all_services = session.get_available_services()
    
    # Categorize services
    compute_services = [s for s in all_services if any(x in s for x in ['ec2', 'lambda', 'batch', 'ecs'])]
    storage_services = [s for s in all_services if any(x in s for x in ['s3', 'dynamodb', 'rds'])]
    ml_services = [s for s in all_services if any(x in s for x in ['sagemaker', 'comprehend', 'rekognition'])]
    networking_services = [s for s in all_services if any(x in s for x in ['vpc', 'route53', 'cloudfront'])]
    
    print(f"Total services: {len(all_services)}")
    print(f"Compute: {len(compute_services)} services")
    print(f"Storage/Database: {len(storage_services)} services") 
    print(f"Machine Learning: {len(ml_services)} services")
    print(f"Networking: {len(networking_services)} services")
    
    # Protocol handling
    print(f"\n=== Protocol Support ===")
    s3_client = session.create_client('s3', region_name='us-east-1')
    ec2_client = session.create_client('ec2', region_name='us-east-1')
    dynamodb_client = session.create_client('dynamodb', region_name='us-east-1')
    
    protocols = {
        'S3': s3_client._service_model.protocol,
        'EC2': ec2_client._service_model.protocol,
        'DynamoDB': dynamodb_client._service_model.protocol
    }
    
    for service, protocol in protocols.items():
        print(f"• {service}: {protocol}")
    
    # Request/Response handling
    print(f"\n=== Request/Response Architecture ===")
    print("• AWSRequest: Custom request object with AWS-specific features")
    print("• Request signing: SigV4, SigV2 authentication protocols")
    print("• Response parsing: Service-specific parsers")
    print("• Error handling: Comprehensive exception hierarchy")
    print("• Retry logic: Exponential backoff with jitter")
    
    # Key features summary
    print(f"\n=== Key Botocore Capabilities ===")
    capabilities = [
        "Direct AWS API mapping with full parameter support",
        "Automatic request signing and authentication",
        "Built-in retry logic with configurable strategies", 
        "Event-driven architecture for customization",
        "Comprehensive error handling and exceptions",
        "Endpoint resolution for all AWS regions",
        "Credential provider chain management",
        "Service model introspection and validation",
        "Paginator support for large result sets",
        "Waiter functionality for resource polling",
        "Thread-safe client implementations",
        "Extensive configuration and customization options"
    ]
    
    for capability in capabilities:
        print(f"✓ {capability}")
    
    print(f"\n=== Relationship to Boto3 ===")
    print("• Botocore: Low-level, protocol-aware AWS client")
    print("• Boto3: High-level, resource-oriented interface")
    print("• Boto3 uses botocore internally for all AWS operations")
    print("• Direct botocore usage provides maximum control and flexibility")
    print("• Both libraries maintained by AWS SDK team")
    
    print("\nBotocore comprehensive architecture overview completed successfully!")
    
except Exception as e:
    print(f"Error in comprehensive architecture overview: {e}")
    raise AssertionError(
        f"botocore example (cell 9) failed: {type(e).__name__}: {e}"
    ) from e